The goal of this notebook is to aggregate relative masses per brain region. 

Output .csvs are different versions where data is normalized depending on what brain regions are to be considered for the relative mass calculations

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import json

from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np
import zarr

from dotenv import dotenv_values
from matplotlib import pyplot as plt

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [3]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

SPATIALDIR = DATADIR / "7_spatial"

OUTDIR = SPATIALDIR / "analyses"
OUTDIR.mkdir(exist_ok=True, parents=True)


In [4]:
with open(str(SPATIALDIR / 'structure_tree_processed.json'), 'r') as f:
    structure_tree = json.load(f)

In [5]:
# Load voxel grid Zarr store
voxel_store = zarr.open(SPATIALDIR / 'sp_voxels.zarr')
voxel_store.tree()

/
├── L1_mass (6, 168, 144, 107) float32
├── L1_prob (6, 168, 144, 107) float32
├── L1_rel_mass (6, 168, 144, 107) float32
├── L2_mass (82, 168, 144, 107) float32
├── L2_prob (82, 168, 144, 107) float32
├── L2_rel_mass (82, 168, 144, 107) float32
├── L3_mass (274, 168, 144, 107) float32
├── L3_prob (274, 168, 144, 107) float32
├── L3_rel_mass (274, 168, 144, 107) float32
├── L4_mass (716, 168, 144, 107) float32
├── L4_prob (716, 168, 144, 107) float32
├── L4_rel_mass (716, 168, 144, 107) float32
├── L5_mass (932, 168, 144, 107) float32
├── L5_prob (932, 168, 144, 107) float32
├── L5_rel_mass (932, 168, 144, 107) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
├── region_labels (168, 144, 107) uint32
└── region_masks (176, 168, 144, 107) bool

### Let's aggregate relative masses per brain region

In [6]:
def get_region_distribution(voxel_store, level: int) -> pd.DataFrame:
    """
    For a given level i, creates a DataFrame of shape (n_labels, n_regions)
    where each entry is the sum of rel_mass for that label within that brain region,
    excluding NaN voxels.
    
    Parameters
    ----------
    voxel_store : zarr Group
        The zarr store containing the arrays
    level : int
        The level i to use (e.g. 1 for L1, 2 for L2, etc.)
    
    Returns
    -------
    pd.DataFrame of shape (n_labels, n_regions)
        index: label ids
        columns: brain region ids
    """
    
    # Load region labels and flatten
    region_labels_flat = voxel_store["region_labels"][:].reshape(-1)  # (nx*ny*nz,)
    unique_regions = np.unique(region_labels_flat)
    
    # Map region ids to column indices
    region_to_idx = {r: i for i, r in enumerate(unique_regions)}
    region_idx = np.array([region_to_idx[r] for r in region_labels_flat])  # (n_voxels,)

    # Get metadata
    label_ids = voxel_store[f"L{level}_rel_mass"].attrs["ids"]
    n_labels = voxel_store[f"L{level}_rel_mass"].shape[0]
    n_regions = len(unique_regions)
    
    # Preallocate result
    result = np.zeros((n_labels, n_regions), dtype=np.float32)
    
    # Iterate over labels one at a time to keep memory low
    for i in range(n_labels):
        # Load single label slice and flatten
        label_mass = voxel_store[f"L{level}_rel_mass"][i].reshape(-1)  # (n_voxels,)
        
        # Mask out NaN voxels
        valid_mask = ~np.isnan(label_mass)
        
        # Only accumulate non-NaN values
        np.add.at(result[i], region_idx[valid_mask], label_mass[valid_mask])

    # Prep output
    df = pd.DataFrame(result, index=label_ids, columns=unique_regions)
    df.drop(columns=0, inplace=True) # remove outside-of-brain indicator region

    return df

In [ ]:
dfs = []
for i in range(1,6):
    dfs.append(get_region_distribution(voxel_store, level = i))
rm_complete = pd.concat(dfs, axis=0)

In [30]:
rm_complete.tail()

,1,6,12,15,23,27,30,38,54,56,...,606826655,606826659,607344830,607344834,607344838,607344846,607344850,607344854,607344858,607344862
NN5-42,0.000607,0.003172,0.000107,0.002833,0.001197,1.097083e-06,0.001730,0.003435,0.000173,0.002590,...,0.000218,0.000378,3.189441e-05,0.000199,2.707334e-05,2.001378e-05,4.405161e-05,9.865806e-10,2.595672e-05,0.000037
NN5-43,0.000863,0.005263,0.000029,0.002889,0.000408,8.531752e-07,0.009600,0.011790,0.000037,0.001337,...,0.000204,0.000092,1.933762e-05,0.000018,1.976424e-07,1.972013e-05,1.289360e-07,1.662997e-07,8.429211e-07,0.000023
NN5-44,0.000392,0.000827,0.000012,0.008250,0.000401,1.515606e-07,0.013941,0.019665,0.000017,0.001740,...,0.000247,0.000199,2.799005e-09,0.000001,0.000000e+00,7.383582e-09,1.306707e-08,0.000000e+00,6.737443e-09,0.000003
NN5-45,0.000682,0.001444,0.000041,0.021460,0.000187,3.825168e-07,0.020138,0.034036,0.000045,0.002931,...,0.000037,0.000091,1.521245e-05,0.000043,2.862022e-06,9.944482e-07,1.432782e-07,4.985125e-09,3.234689e-07,0.000015
NN5-46,0.000163,0.000925,0.000002,0.025869,0.000758,0.000000e+00,0.003541,0.008895,0.000099,0.003360,...,0.000896,0.000248,2.497660e-04,0.000003,0.000000e+00,1.541680e-07,1.267680e-08,9.272888e-08,2.776072e-06,0.000071


In [31]:
rm_complete.to_csv(OUTDIR / "regionwise_rm_all.csv")

### Let's create region metadata that we can use to filter the above relative masses for specific purposes

In [32]:
hy_ids = []
non_hy_ids = []

hy_acros = []
non_hy_acros = []

hy_names = []
non_hy_names = []

id_to_acro_mapper = {}
id_to_name_mapper = {}

for ID in structure_tree.keys():
    structure_path = structure_tree[ID]["path"]
    acro = structure_tree[ID]["acronym"]
    name = structure_tree[ID]["name"]

    id_to_acro_mapper[int(ID)] = acro
    id_to_name_mapper[int(ID)] = name

    if (ID != "1097") & (1097 not in structure_path): #HY id
        non_hy_ids.append(int(ID))
        non_hy_acros.append(acro)
        non_hy_names.append(name)
    else:
        hy_ids.append(int(ID))
        hy_acros.append(acro)
        hy_names.append(name)

In [33]:
region_meta = pd.concat([
    pd.DataFrame({"ID": hy_ids, "acronym": hy_acros, "name": hy_names, "descends_from_HY": [True] * len(hy_names)}),
    pd.DataFrame({"ID": non_hy_ids, "acronym": non_hy_acros, "name": non_hy_names, "descends_from_HY": [False] * len(non_hy_names)})
])
region_meta["is_in_bbox"] = False
region_meta.loc[region_meta.ID.isin(rm_complete.columns), "is_in_bbox"] = True

In [34]:
region_meta

,ID,acronym,name,descends_from_HY,is_in_bbox
0,1097,HY,Hypothalamus,True,True
1,10671,ME,Median eminence,True,True
2,290,LZ,Hypothalamic lateral zone,True,False
3,797,ZI,Zona incerta,True,True
4,804,FF,Fields of Forel,True,True
...,...,...,...,...,...
1231,526322264,FRP6b,"Frontal pole, layer 6b",False,False
1232,526157196,FRP6a,"Frontal pole, layer 6a",False,False
1233,526157192,FRP5,"Frontal pole, layer 5",False,False
1234,667,FRP2/3,"Frontal pole, layer 2/3",False,False


In [35]:
region_meta.to_csv(SPATIALDIR / "abc_region_metadata.csv", index=False)

### Let's create a version where the "root" region is filtered out

In [36]:
rm_wo_root = rm_complete.loc[:, [ID for ID in rm_complete.columns if ID != 997]]
# norm to 1 
rm_wo_root = rm_wo_root.div(rm_wo_root.sum(axis=1), axis=0)
rm_wo_root.columns = [id_to_acro_mapper[ID] for ID in rm_wo_root.columns]

In [37]:
rm_wo_root.head()

,TMv,int,IF,PT,AAA,IGL,PVa,PVH,mfb,ACB,...,MMp,MMd,PN,IPR,IPC,IPL,IPI,IPDM,IPDL,IPRL
Gb1-2,0.001161,0.011962,0.000149,0.000669,0.008502,0.000025,0.001321,0.001105,0.000140,0.060894,...,0.000014,0.000133,0.000041,0.000380,0.000033,0.000051,0.000042,4.281577e-07,0.000042,0.000097
Gl1-1,0.001100,0.010536,0.001118,0.006645,0.005775,0.000036,0.000748,0.007419,0.001143,0.003039,...,0.002030,0.002900,0.000129,0.001204,0.000074,0.000118,0.000074,6.246938e-06,0.000171,0.000400
NN1-1,0.000730,0.062364,0.000815,0.002909,0.004541,0.000047,0.000348,0.001283,0.000916,0.009575,...,0.000261,0.000638,0.000264,0.001208,0.000190,0.000200,0.000138,1.263860e-05,0.000152,0.000336
NN1-2,0.000748,0.018538,0.000461,0.002654,0.007587,0.000032,0.001068,0.005545,0.000702,0.016102,...,0.000465,0.000597,0.000228,0.001564,0.000203,0.000472,0.000135,1.460839e-06,0.000218,0.000228
NN1-3,0.004468,0.013978,0.000839,0.004715,0.010810,0.000043,0.000571,0.003940,0.000787,0.013912,...,0.001111,0.003517,0.000163,0.001928,0.000274,0.000385,0.000108,3.884028e-06,0.000141,0.000465


In [38]:
rm_wo_root.to_csv(OUTDIR / "regionwise_rm_all_wo_root.csv")

### Let's create a version where both HY and non-HY regions are allowed but fiber tracts are filtered out 

In [39]:
tracts = region_meta.loc[region_meta.acronym.str.islower(), "acronym"].to_list()
#tracts

In [40]:
rm_wo_tracts = rm_wo_root.loc[:, [c for c in rm_wo_root.columns if c not in tracts]]
# norm to 1 
rm_wo_tracts = rm_wo_tracts.div(rm_wo_tracts.sum(axis=1), axis=0)

In [41]:
rm_wo_tracts.head()

,TMv,IF,PT,AAA,IGL,PVa,PVH,ACB,MT,IMD,...,MMp,MMd,PN,IPR,IPC,IPL,IPI,IPDM,IPDL,IPRL
Gb1-2,0.001216,0.000157,0.000701,0.008910,0.000026,0.001384,0.001158,0.063813,0.000342,0.000438,...,0.000014,0.000140,0.000043,0.000398,0.000034,0.000053,0.000044,4.486816e-07,0.000044,0.000102
Gl1-1,0.001175,0.001195,0.007099,0.006170,0.000039,0.000799,0.007925,0.003247,0.000475,0.005751,...,0.002169,0.003098,0.000138,0.001287,0.000079,0.000126,0.000080,6.673851e-06,0.000183,0.000428
NN1-1,0.000898,0.001003,0.003578,0.005585,0.000058,0.000428,0.001578,0.011777,0.001094,0.001363,...,0.000321,0.000784,0.000325,0.001486,0.000234,0.000245,0.000170,1.554499e-05,0.000186,0.000413
NN1-2,0.000809,0.000499,0.002872,0.008208,0.000034,0.001155,0.005999,0.017420,0.000811,0.001375,...,0.000503,0.000646,0.000247,0.001692,0.000219,0.000510,0.000146,1.580479e-06,0.000236,0.000246
NN1-3,0.004885,0.000917,0.005156,0.011820,0.000047,0.000624,0.004308,0.015211,0.001067,0.003366,...,0.001214,0.003845,0.000178,0.002108,0.000300,0.000421,0.000118,4.246746e-06,0.000154,0.000509


In [42]:
rm_wo_tracts.to_csv(OUTDIR / "regionwise_rm_all_wo_root_or_fibers.csv")

### Let's create a version of the above restricted to HY descending brain regions

In [43]:
rm_wo_tracts_hy = rm_wo_tracts.loc[:, [c for c in rm_wo_tracts.columns if c in hy_acros]]
# norm to 1 
rm_wo_tracts_hy = rm_wo_tracts_hy.div(rm_wo_tracts_hy.sum(axis=1), axis=0)

In [44]:
rm_wo_tracts_hy.head()

,TMv,PVa,PVH,PVHd,ADP,AHN,PVi,PVp,PVpo,RCH,...,HY,PS,TMd,ME,VMPO,PeF,MMl,MMm,MMp,MMd
Gb1-2,0.006751,0.007684,0.006426,0.009450,0.011216,0.058012,0.010843,0.010975,0.019578,0.008290,...,0.188627,0.012184,0.002222,0.007548,0.005155,0.012813,0.000732,0.000384,0.000079,0.000775
Gl1-1,0.003459,0.002353,0.023333,0.006919,0.003396,0.039845,0.010897,0.009214,0.012308,0.006636,...,0.162567,0.001840,0.004350,0.003064,0.002292,0.007933,0.030843,0.024505,0.006384,0.009122
NN1-1,0.003236,0.001543,0.005688,0.008262,0.006640,0.039844,0.006289,0.003013,0.005629,0.010631,...,0.193250,0.003886,0.000931,0.002591,0.002832,0.021743,0.008567,0.004741,0.001156,0.002827
NN1-2,0.002935,0.004189,0.021761,0.012229,0.009201,0.060077,0.009094,0.004104,0.011354,0.009444,...,0.191273,0.009402,0.000389,0.006105,0.003450,0.015942,0.004506,0.003004,0.001826,0.002342
NN1-3,0.019835,0.002536,0.017493,0.008756,0.005271,0.032650,0.008483,0.007614,0.008535,0.006460,...,0.166561,0.005189,0.001709,0.013722,0.002015,0.011433,0.019457,0.010107,0.004931,0.015613


In [45]:
rm_wo_tracts_hy.to_csv(OUTDIR / "regionwise_rm_HY_wo_root_or_fibers.csv")

### Let's create versions without HY itself to focus on the actual brain nuclei rather than the parenchyma

only HY

In [46]:
rm_wo_tracts_hy_wo_hyacro = rm_wo_tracts_hy.loc[:, [c for c in rm_wo_tracts_hy.columns if c != "HY"]]
# norm to 1 
rm_wo_tracts_hy_wo_hyacro = rm_wo_tracts_hy_wo_hyacro.div(rm_wo_tracts_hy.sum(axis=1), axis=0)

In [47]:
rm_wo_tracts_hy_wo_hyacro.head()

,TMv,PVa,PVH,PVHd,ADP,AHN,PVi,PVp,PVpo,RCH,...,PMv,PS,TMd,ME,VMPO,PeF,MMl,MMm,MMp,MMd
Gb1-2,0.006751,0.007684,0.006426,0.009450,0.011216,0.058012,0.010843,0.010975,0.019578,0.008290,...,0.008040,0.012184,0.002222,0.007548,0.005155,0.012813,0.000732,0.000384,0.000079,0.000775
Gl1-1,0.003459,0.002353,0.023333,0.006919,0.003396,0.039845,0.010897,0.009214,0.012308,0.006636,...,0.019997,0.001840,0.004350,0.003064,0.002292,0.007933,0.030843,0.024505,0.006384,0.009122
NN1-1,0.003236,0.001543,0.005688,0.008262,0.006640,0.039844,0.006289,0.003013,0.005629,0.010631,...,0.005753,0.003886,0.000931,0.002591,0.002832,0.021743,0.008567,0.004741,0.001156,0.002827
NN1-2,0.002935,0.004189,0.021761,0.012229,0.009201,0.060077,0.009094,0.004104,0.011354,0.009444,...,0.006180,0.009402,0.000389,0.006105,0.003450,0.015942,0.004506,0.003004,0.001826,0.002342
NN1-3,0.019835,0.002536,0.017493,0.008756,0.005271,0.032650,0.008483,0.007614,0.008535,0.006460,...,0.010136,0.005189,0.001709,0.013722,0.002015,0.011433,0.019457,0.010107,0.004931,0.015613


In [48]:
rm_wo_tracts_hy_wo_hyacro.to_csv(OUTDIR / "regionwise_rm_HY_wo_root_or_fibers_or_HYacro.csv")

HY and non-HY regions allowed

In [49]:
rm_wo_tracts_wo_hyacro = rm_wo_tracts.loc[:, [c for c in rm_wo_tracts.columns if c != "HY"]]
# norm to 1 
rm_wo_tracts_wo_hyacro = rm_wo_tracts_wo_hyacro.div(rm_wo_tracts_wo_hyacro.sum(axis=1), axis=0)

In [50]:
rm_wo_tracts_wo_hyacro.head()

,TMv,IF,PT,AAA,IGL,PVa,PVH,ACB,MT,IMD,...,MMp,MMd,PN,IPR,IPC,IPL,IPI,IPDM,IPDL,IPRL
Gb1-2,0.001259,0.000162,0.000726,0.009223,0.000027,0.001433,0.001198,0.066058,0.000354,0.000454,...,0.000015,0.000144,0.000045,0.000412,0.000036,0.000055,0.000045,4.644644e-07,0.000045,0.000106
Gl1-1,0.001244,0.001265,0.007514,0.006530,0.000041,0.000846,0.008389,0.003437,0.000502,0.006087,...,0.002295,0.003279,0.000146,0.001362,0.000084,0.000134,0.000084,7.063912e-06,0.000194,0.000453
NN1-1,0.000949,0.001060,0.003780,0.005901,0.000062,0.000452,0.001667,0.012444,0.001155,0.001440,...,0.000339,0.000829,0.000344,0.001570,0.000248,0.000259,0.000179,1.642550e-05,0.000197,0.000436
NN1-2,0.000854,0.000527,0.003032,0.008665,0.000036,0.001219,0.006333,0.018390,0.000856,0.001451,...,0.000531,0.000682,0.000260,0.001787,0.000231,0.000539,0.000154,1.668460e-06,0.000249,0.000260
NN1-3,0.005094,0.000956,0.005376,0.012325,0.000049,0.000651,0.004492,0.015862,0.001113,0.003510,...,0.001266,0.004010,0.000186,0.002198,0.000312,0.000439,0.000124,4.428398e-06,0.000161,0.000530


In [51]:
rm_wo_tracts_wo_hyacro.to_csv(OUTDIR / "regionwise_rm_all_wo_root_or_fibers_or_HYacro.csv")